In [1]:
%%writefile cleaner.py
import pandas as pd
import re
import os
from fractions import Fraction
import ast
import json
import uuid
import unicodedata
from config import DATA_PATH, EDA_PATH

def check_malformed_literal(df, col_name):
    """
    检查指定列中哪些行无法被 ast.literal_eval 解析。
    """
    print(f"检查列是否符合列表格式: {col_name} ...")
    error_count = 0
    
    for index, value in df[col_name].items():
        # 只处理字符串类型，因为 ast.literal_eval 只接受字符串
        if isinstance(value, str):
            try:
                ast.literal_eval(value)
            except (ValueError, SyntaxError) as e:
                error_count += 1
                print("-" * 30)
                print(f"Index: {index}")
                print(f"Error Type: {type(e).__name__}")
                print(f"Content: {repr(value)}") # 使用 repr 打印出原始字符串（包括空格和换行）

def process_fractions_dynamic(text_to_process):
    if not isinstance(text_to_process, str):
        return str(text_to_process)
    # Step0: 预处理
    # --- 0.1 清理数值边界的特殊标记符号 ---
    # (123) [123] ˹123˺ ⌈123⌋ 「123」 ⸢123⸣ → 123
    text_to_process = re.sub(r'\((\+?\d+)\)', r'\1', text_to_process)
    text_to_process = re.sub(r'\[(\+?\d+)\]', r'\1', text_to_process)
    text_to_process = re.sub(r'˹(\+?\d+)˺', r'\1', text_to_process)
    text_to_process = re.sub(r'⌈(\+?\d+)⌋', r'\1', text_to_process)
    text_to_process = re.sub(r'「(\+?\d+)」', r'\1', text_to_process)
    text_to_process = re.sub(r'⸢(\+?\d+)⸣', r'\1', text_to_process)

    # --- 0.2 转换 X+X 格式的数值：循环处理直到没有更多加法运算需要处理 ---
    def calculate_addition(match):
        try:
            val1 = float(match.group(1))
            val2 = float(match.group(2))
            result = val1 + val2
            # 格式化逻辑: 如果是整数则转int str，否则保留有效小数
            return "{:.5f}".format(result).rstrip('0').rstrip('.') if result % 1 != 0 else str(int(result))
        except ValueError:
            return match.group(0)
    addition_pattern = r'(\d+(?:\.\d+)?)\s*\+\s*(\d+(?:\.\d+)?)'
    while True:
        new_text = re.sub(addition_pattern, calculate_addition, text_to_process)
        if new_text == text_to_process:
            break
        text_to_process = new_text

    # --- 0.3 月份替换 ---
    # Month I - month XII -> month 1 - month 12
    roman_to_arabic = {
        "I": "1", "II": "2", "III": "3", "IV": "4", "V": "5", "VI": "6",
        "VII": "7", "VIII": "8", "IX": "9", "X": "10", "XI": "11", "XII": "12"
    }
    roman_pattern = r'\b(Month|month)\s+(VIII|XII|VII|III|XI|IX|VI|IV|II|X|V|I)\b'
    def replace_roman(match):
        return f"{match.group(1)} {roman_to_arabic[match.group(2)]}"
    text_to_process = re.sub(roman_pattern, replace_roman, text_to_process)

    # Step1: 转换 X / X 格式的分数
    # --- 1.1 分数转换：规则内的分数转换到 Unicode 分数符号 ---
    fraction_map = {
        r"1\s*/\s*6": "⅙",
        r"1\s*/\s*4": "¼",
        r"1\s*/\s*3": "⅓",
        r"1\s*/\s*2": "½",
        r"2\s*/\s*3": "⅔",
        r"3\s*/\s*4": "¾",
        r"5\s*/\s*6": "⅚",
        # r"1\s*/\s*8": "⅛", r"3\s*/\s*8": "⅜", r"5\s*/\s*8": "⅝", r"7\s*/\s*8": "⅞",
    }
    for pattern, char in fraction_map.items():
        text_to_process = re.sub(pattern, char, text_to_process)

    # --- 1.2 通用数值分数转换：剩余的分数形式转换为小数格式 ---
    def calc_fraction(match):
        groups = match.groups()
        if groups[0]: # 如果匹配到了整数部分 (例如 "5 11/12")
            integer_part = int(groups[0])
            num = int(groups[1])
            den = int(groups[2])
            val = integer_part + (num / den)
        else: # 只匹配到了分数部分 (例如 "11/12")
            num = int(groups[3])
            den = int(groups[4])
            val = num / den
        # 格式化为5位小数 (例如 5.916666... -> 5.91667)
        return "{:.5f}".format(val).rstrip('0').rstrip('.') if val % 1 != 0 else str(int(val))
    # 模式1 (混合分数): (\d+)\s+(\d+)/(\d+) -> 匹配 "5 11/12"  |  模式2 (纯分数): (\d+)/(\d+) -> 匹配 "11/12"
    fraction_pattern = r'(\d+)\s+(\d+)/(\d+)|(\d+)/(\d+)'   
    text_to_process = re.sub(fraction_pattern, calc_fraction, text_to_process)
    
    # Step2: 转换小数格式的分数到 Unicode 分数符号
    # --- 2.1 处理长浮点数损坏数值 ---
    # 正则逻辑：匹配 数字 + 小数点 + 至少5位数字
    # \d+\.\d{5,} 匹配如 1.3333300000000001
    # lambda 表达式将其截断至小数点后 5 位
    text_to_process = re.sub(r'(\d+\.\d{5})\d+', r'\1', text_to_process)

    # --- 2.2 统一成 UNICODE 分数符号 ---
    fraction_lookup = {
            (1, 6): "⅙", # .166
            (1, 4): "¼", # .25
            (1, 3): "⅓", # .333
            (1, 2): "½", # .5
            (2, 3): "⅔", # .666
            (3, 4): "¾", # .75
            (5, 6): "⅚", # .833
        #    (1, 5): "⅕", (2, 5): "⅖", (3, 5): "⅗", (4, 5): "⅘",
        #    (1, 8): "⅛", (3, 8): "⅜", (5, 8): "⅝", (7, 8): "⅞"
    }
    def replacer(match):
        full_str = match.group(0)
        try:
            val = float(full_str)
            integer_part = int(val)
            decimal_part = val - integer_part
            # Return integer if decimal is negligible
            if decimal_part < 0.0001: 
                return str(integer_part)
            # Limit denominator handles 0.3333 -> 1/3
            frac = Fraction(decimal_part).limit_denominator(12) 

            unicode_frac = fraction_lookup.get((frac.numerator, frac.denominator))
            if unicode_frac:
                if integer_part == 0:
                    return unicode_frac
                else:
                    # Current logic: Space between Integer and Fraction
                    return f"{integer_part} {unicode_frac}"

            return full_str 
        except ValueError:
            return full_str
    processed_text = re.sub(r'\b\d+\.\d+\b', replacer, text_to_process)
    
    # Step3: Robustness check
    # --- 3.1 保证带分数中间有空格 ---
    processed_text = re.sub(r'(\d)([¼½¾⅓⅔⅕⅖⅗⅘⅙⅚⅛⅜⅝⅞])', r'\1 \2', processed_text)
    
    return processed_text

def preprocess_transliteration(text):
    if pd.isna(text):
        return ""
    # --- 0. oare 数据特殊符号移除 ---
    pattern = r"\s*Seal Impression [A-Z]\s*"
    text = re.sub(pattern, "", text)
    pattern = r"Seal Impression"
    text = re.sub(pattern, "", text)
    pattern = r"broken"
    text = re.sub(pattern, "xxx", text)
    pattern = r"------------"
    text = re.sub(pattern, "", text)
    pattern = r"---- Single Ruling ----"
    text = re.sub(pattern, "", text)
    pattern = r"---- Double Ruling ----"
    text = re.sub(pattern, "", text)
    
    # --- 1. 字符替换与括号处理 ---
    char_map = str.maketrans({
        "X": "x",
        "×": "x",
        "ā": "a",
        "Ş": "Ṣ",
        "ş": "ṣ",
        "Ș": "Ṣ",
        "ș": "ṣ",
        "Ț": "Ṭ",
        "ț": "ṭ",
        "İ": "I",
        "Ī": "I",
        "ī": "i",
        "Î": "I",
        "î": "i",
        "ı": "i",
        "h": "ḫ",
        "H": "Ḫ",
        "ḥ": "ḫ",
        "Ḥ": "Ḫ",
        "=": "-",
        "(": "{",  # for deterministic
        ")": "}",
        "—": "-",
        "–": "-",
    })
    text = text.translate(char_map)
    text = text.replace("ᵈ", r"{d}").replace("ᵏⁱ", r"{ki}")   # 手动操作
    text = text.replace(r"{{", "{").replace(r"}}", "}")
    # text = text.replace("lá+lá", "lá-lá").replace("la+la", "la-la")
    text = text.replace("??", "?").replace("!!", "!").replace("//", "/")
    text = text.replace("-/", "-")
    text = text.replace("{?}", "").replace("{!}", "")
    text = text.replace("(?)", "").replace("(!)", "")
    # Remove bad characters
    upper_chars = ["⁰","¹","²","³","⁴","⁵","⁶","⁷","⁸","⁹","⁻","°", "˚"]
    for ch in upper_chars:
        text = text.replace(ch, "")
    bad_chars = """"ʺʾ’´ʼˊ˘ˋˇˈʿʹ'`^˹˺⸢⸣⌜⌝「」⌈⌋⌊⌉˻˼⸤⸥≪≫《》⟪⟫«»〈〉⟨⟩˃‹›⁽⁾ˁˀ"""
    for ch in bad_chars:
        text = text.replace(ch, "")
    bad_chars = """!?⸮ʔ⸮,/;˷˴|:⁺*"""  # +
    for ch in bad_chars:
        text = text.replace(ch, " ")

    # --- 2.1 处理数值符号 ---
    text = process_fractions_dynamic(text)
    
    # --- 2. 角标统一处理 ---
    # 4 a-bi4-a 0.5 2+10 ù a-šùr-DU10: 4 a-bi₄-a 0.5 2+10 ù a-šùr-DU₁₀
    # 定义允许的字母范围（包含普通字母和特殊字符）
    letters_str = "a-zA-ZšŠṣṢṭṬḫḪ"
    # 定义数字到角标的映射表
    subscript_trans = str.maketrans("0123456789", "₀₁₂₃₄₅₆₇₈₉")
    # 正则逻辑：匹配紧跟在字母后面的数字
    def replace_group_with_subscript(match):
        digits = match.group(0)
        return digits.translate(subscript_trans)
    text = re.sub(f"(?<=[{letters_str}])\d+", replace_group_with_subscript, text)

    # --- 2. 建议替换的标注 ---
    # ['(fem.)', '(sing.)', '(pl.)', '(plural)', '(f.)', '(plur.)'] -> ''
    pattern = r'\(\s*(?:fem\.|sing\.|pl\.|plural|f\.|plur\.)(?:\s+(?:fem\.|sing\.|pl\.|plural|f\.|plur\.))*\s*\)'
    text = re.sub(pattern, '', text)
    #  ['(K)', '(Rs)', '(lR)'] -> ''
    pattern = r'\(\s*(?:K|Rs|lR)(?:\s+(?:K|Rs|lR))*\s*\)'
    text = re.sub(pattern, '', text)
    # ['le.e.', 'lo.e.', 'l.o.e.', 'obv.', 'rev.', 'r.e.', 'u.e.'] -> ''
    pattern = r'\b(?:le\.e\.|lo\.e\.|l\.o\.e\.|obv\.|rev\.|r\.e\.|u\.e\.)\s*'
    text = re.sub(pattern, '', text)

    # --- 3. Gap替换 ---
    # ['PN'/'Pn'] -> '<gap>'
    pattern = r'\bPN\b|\bPn\b'
    text = re.sub(pattern, '<gap>', text)
    # break/broken -> ''
    text = re.sub(r'\bbreak\b|\bbroken\b', '', text)
    # [] -> [xxx]
    text = re.sub(r'\[[\sx?]*\]', '[xxx]', text)
    bad_chars = "[]"
    for ch in bad_chars:
        text = text.replace(ch, "")
    # Multi-dot patterns → <gap>
    text = re.sub(r'(?:\.\s+){2,}\.', lambda m: m.group(0).replace(' ', ''), text)
    text = re.sub(r'\.{3,}(\s+\.{3,})*', '<gap>', text)
    text = re.sub(r'……|…', '<gap>', text)
    # x patterns → <gap>
    text = re.sub(r'xx+', '<gap>', text)
    # 处理孤立的 x (前后是空格或边界)
    text = re.sub(r'\bx\b', '<gap>', text)
    # 直接替换 {large break}
    text = text.replace("{large break}", "<gap>")
    # 正则替换 {N broken lines} (例如 {3 broken lines}, {4 broken lines})
    text = re.sub(r'\{\d+\s+broken\s+lines\}', '<gap>', text)
    # Temporarily protect tokens from cleanup
    text = text.replace("<gap>", "\x00GAP\x00")
    # Remove bad characters
    bad_chars = "<>"
    for ch in bad_chars:
        text = text.replace(ch, "")
    # Restore tokens
    text = text.replace("\x00GAP\x00", " <gap> ")
    
    # Normalize whitespace and trim
    text = re.sub(r"\s+", " ", text).strip().strip("-")
    text = text.replace(" -", "-").replace("- ", "-")
    text = text.replace(" .", ".").replace(". ", ".")
    text = text.replace("> .", ">.").replace("> :", ">:").replace("> ’", ">’")

    # merge <gap>
    pattern = r'<gap>([ \-\t]*<gap>)+'
    text = re.sub(pattern, '<gap>', text)
    text = text.replace("{ <gap> }", "<gap>")
    text = text.replace("[ <gap> ]", "<gap>")
    
    return text

def preprocess_train_translation(text):
    if pd.isna(text):
        return ""
    text = str(text)
    
    # --- 1. 字符替换与括号处理 ---
    char_map = str.maketrans({
        "X": "x",
        "ʾ": "'",
        "Ş": "Ṣ",
        "ş": "ṣ",
        "ș": "ṣ",
        "Ț": "Ṭ",
        "ț": "ṭ",
        "ḫ": "h",
        "Ḫ": "H",
        "ḥ": "ḫ",
        "Ḥ": "Ḫ",
    })
    text = text.translate(char_map)
    text = text.replace("{?}", "").replace("{!}", "")
    text = text.replace("(?)", "").replace("(!)", "")
    # Remove bad characters
    upper_chars = ["⁰","¹","²","³","⁴","⁵","⁶","⁷","⁸","⁹","⁻","°","˚"]
    for ch in upper_chars:
        text = text.replace(ch, "")
    lower_chars = ["₀","₁","₂","₃","₄","₅","₆","₇","₈","₉"]
    for ch in lower_chars:
        text = text.replace(ch, "")
    bad_chars = """"ʺˊ˘ˋˇˈʿ`^˹˺⸢⸣⌜⌝「」⌈⌋⌊⌉˻˼⸤⸥≪≫《》⟪⟫«»〈〉⟨⟩˃‹›⁽⁾ˁˀ"""
    for ch in bad_chars:
        text = text.replace(ch, "")
    bad_chars = """ʹ'ʾ´ʼ"""
    for ch in bad_chars:
        text = text.replace(ch, "’")

    # --- 2.1 处理数值符号 ---
    text = process_fractions_dynamic(text)

    # --- 2.2 建议替换的标注 ---
    # ['(fem.)', '(sing.)', '(pl.)', '(plural)', '(f.)', '(plur.)'] -> ''
    pattern = r'\(\s*(?:fem\.|sing\.|pl\.|plural|f\.|plur\.)(?:\s+(?:fem\.|sing\.|pl\.|plural|f\.|plur\.))*\s*\)'
    text = re.sub(pattern, '', text)
    #  ['(K)', '(Rs)', '(lR)'] -> ''
    pattern = r'\(\s*(?:K|Rs|lR)(?:\s+(?:K|Rs|lR))*\s*\)'
    text = re.sub(pattern, '', text)
    # ['le.e.', 'lo.e.', 'l.o.e.', 'obv.', 'rev.', 'r.e.', 'u.e.'] -> ''
    pattern = r'\b(?:le\.e\.|lo\.e\.|l\.o\.e\.|obv\.|rev\.|r\.e\.|u\.e\.)\s*'
    text = re.sub(pattern, '', text)

    # --- 3. Gap替换 ---
    # ['PN'] -> '<gap>'
    pattern = r'\bPN\b|\bPn\b'
    text = re.sub(pattern, '<gap>', text)
    # [] -> [xxx]
    text = re.sub(r'\[[\sx?]*\]', '[xxx]', text)
    # Multi-dot patterns → <gap>
    text = re.sub(r'(?:\.\s+){2,}\.', lambda m: m.group(0).replace(' ', ''), text)
    text = re.sub(r'\.{3,}(\s+\.{3,})*', '<gap>', text)
    text = re.sub(r'……|…', '<gap>', text)
    # x patterns → <gap>
    text = re.sub(r'xx+', '<gap>', text, flags=re.I)
    # 处理孤立的 x (前后是空格或边界)
    text = re.sub(r'\bx\b', '<gap>', text, flags=re.I)
    # 直接替换 {large break}
    text = text.replace("{large break}", "<gap>")
    # 正则替换 {N broken lines} (例如 {3 broken lines}, {4 broken lines})
    text = re.sub(r'\{\d+\s+broken\s+lines\}', '<gap>', text)
    # Temporarily protect tokens from cleanup
    text = text.replace("<gap>", "\x00GAP\x00")
    # Remove bad characters
    bad_chars = '˹˺⸢⸣「」⌈⌋⌊|'  # []
    for ch in bad_chars:
        text = text.replace(ch, "")
    bad_chars = "/*+"
    for ch in bad_chars:
        text = text.replace(ch, " ")
    # Restore tokens
    text = text.replace("\x00GAP\x00", " <gap> ")

    # Normalize whitespace and trim
    text = re.sub(r"\s+", " ", text).strip().strip("-")
    text = text.replace(" -", "-").replace("- ", "-")
    text = text.replace("> .", ">.").replace("> :", ">:")

    # merge <gap>
    pattern = r'<gap>([ \-\t]*<gap>)+'
    text = re.sub(pattern, '<gap>', text)
    text = text.replace("{ <gap> }", "<gap>")
    text = text.replace("[ <gap> ]", "<gap>")
    
    return text

if __name__ == "__main__":
    data_dir = DATA_PATH
    df_path = os.path.join(data_dir, "synth_df0308_en.csv")
    df_target_lang_code = 'en'
    TRANSLITERATION_COL = 'transliteration'
    TRANSLATION_COL = f'{df_target_lang_code}_translation'
    output_path = os.path.join(data_dir, "pipeline/train_synth_akka2en-sentlevel0308.csv")

    df = pd.read_csv(df_path, dtype={'chapter_id': str})
    print(f"Read df: {df.shape} <- {df_path}")
    df = df.dropna(subset=[TRANSLATION_COL])
    print(f"After dropna: {df.shape}")
    
    cols_to_fix = ['line_range', TRANSLITERATION_COL, TRANSLATION_COL]
    for col in cols_to_fix:
        check_malformed_literal(df, col)
        # 使用 ast.literal_eval 安全地解析字符串列表
        df[col] = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

    # Check for length mismatch before explode
    for idx, row in df.iterrows():
        l1 = len(row['line_range']) if isinstance(row['line_range'], list) else 0
        l2 = len(row[TRANSLITERATION_COL]) if isinstance(row[TRANSLITERATION_COL], list) else 0
        l3 = len(row[TRANSLATION_COL]) if isinstance(row[TRANSLATION_COL], list) else 0
        if not (l1 == l2 == l3):
            print(f"Mismatch found - oare_id: {row.get('oare_id', 'Unknown')}, lengths: line_range={l1}, transliteration={l2}, translation={l3}")

    df = df.explode(cols_to_fix)
    print(f"After explode: {df.shape}")

    # df[TRANSLITERATION_COL] = df[TRANSLITERATION_COL].apply(process_fractions_dynamic)
    # df[TRANSLATION_COL] = df[TRANSLATION_COL].apply(process_fractions_dynamic)
    df[TRANSLITERATION_COL] = df[TRANSLITERATION_COL].apply(preprocess_transliteration)
    df[TRANSLATION_COL] = df[TRANSLATION_COL].apply(preprocess_train_translation)
    df[f'{TRANSLITERATION_COL}_char_length'] = df.apply(lambda row: len(row[TRANSLITERATION_COL]), axis=1)
    df[f'{TRANSLATION_COL}_char_length'] = df.apply(lambda row: len(row[TRANSLATION_COL]), axis=1)

    # df = df.drop_duplicates(subset=[TRANSLITERATION_COL, TRANSLATION_COL])
    df['sentence_id'] = df.groupby(['publication_name', 'chapter_id']).cumcount() + 1
    print(f"Save to csv: {df.shape} -> {output_path}")
    df.to_csv(output_path, index=False)

    # df = pd.read_csv(output_path)
    # print(f"Read df: {df.shape} <- {output_path}")
    # # 统计字符，写成json文件
    # for col in [TRANSLATION_COL, TRANSLITERATION_COL]:
    #     unique_chars_set = set()
    #     for text in df[col].fillna("").astype(str):
    #         unique_chars_set.update(list(text))
    #     unique_chars_list = sorted(list(unique_chars_set))
    #     char_eda_path = os.path.join(EDA_PATH, f"chars_in_train_ocr_akka2en-sentlevel0318_{col}.json")
    #     with open(char_eda_path, "w", encoding="utf-8") as f:
    #         json.dump(unique_chars_list, f, ensure_ascii=False, indent=4)
    #     print(f"Save char eda: {char_eda_path}")


Writing cleaner.py
